# 02 · Stage (c) multi-patch STMap HR + abstain (C3 PatchAttnSpec) · fast.ai on a Kaggle T4

**Goal.** Train the shootout's winning learned estimator, C3 `PatchAttnSpec`, on the
pre-generated 16-patch STMap cache, calibrate its abstention cut on val, and export the
**conv-only** static INT8 ONNX that the CPU Hugging Face Space and the phone load.

* Input: `(6, 16, 300)` STMap: RGB/mean-1 and YUV, 16 face patches, 30 s at 10 Hz
  (`mp_data.make_stmap`, vendored verbatim in `learning/hf_space/app_gradio_space.py`).
* Output: 142 logits: 141 one-BPM bins (40-180 BPM) + one `no_pulse/abstain` class.
* Loss: soft-target cross-entropy against a Gaussian (sigma 1.5 BPM) bin target.
* Decode: soft-argmax within +-3 BPM of the best bin; confidence = bin mass in that window.
* Kaggle settings: **GPU T4 x1**, internet on for the pinned install, cache dataset attached.
  14 epochs take about 2 minutes on a T4 (vs ~30 min on 4 CPU cores).

Not a medical device. Synthetic data only.

## Shootout §6 lessons applied here

| # | Lesson (FastAI-CV-Shootout-Report §6) | Where in this notebook |
|---|---|---|
| L1 | Pre-generate the data cache once (the 9,900-session cache is ~1.3 GB float16 `.npz`) and attach it as a Kaggle dataset; never regenerate inside a GPU session | §1 Data |
| L2 | Keep the float16 tensors resident on the GPU, batch 256 with the LR scaled up, `num_workers=0` | §2 DataLoaders |
| L3 | No `vision_learner`, no `aug_transforms`, no flips or crops: the axes are time and frequency, not pixels | §2 DataLoaders, §3 Model |
| L4 | `to_fp16()` mixed precision, but keep spectral maths (DFT / power / log) in FP32 with `torch.autocast(enabled=False)` | §3 Model |
| L5 | `pretrained=False` everywhere: ImageNet weights do not transfer to spectrograms or STMaps | §3 Model |
| L6 | Pin versions (torch 2.15.0, fastai 2.8.12, timm 1.0.31, onnx 1.23.2, onnxruntime 1.29.0, Python 3.13) and export with `torch.onnx.export(dynamo=False, opset_version=17)` | §0 Setup, §5 Export |
| L7 | Budget epochs from measured speed (C3: ~2 min/epoch on CPU, < 10 s/epoch on a T4); static QDQ INT8 for convnets, never dynamic INT8 (5-30x slower on x86) | §4 Train, §5 Export |

## 0 · Setup (L6)

In [ ]:
# L6: pin versions. Kaggle images move under you; a pinned install makes reruns comparable.
!pip install -q torch==2.15.0 fastai==2.8.12 timm==1.0.31 onnx==1.23.2 onnxruntime==1.29.0

In [ ]:
import os, sys, time, json, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./out")
OUT.mkdir(parents=True, exist_ok=True)
print(sys.version.split()[0], torch.__version__, DEVICE,
      torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
# SMOKE=1 runs every cell end to end on tiny random tensors (CPU is fine); use it before a GPU run.
SMOKE = os.environ.get("SMOKE", "0") == "1"

## 1 · Data (L1)

In [ ]:
# L1: build the cache once with mp_data.build_split and attach it as a Kaggle dataset.
CACHE = Path(os.environ.get("PRANASCAN_CACHE", "/kaggle/input/pranascan-mp-cache"))
FILES = {"train": "train_600.npz", "val": "val_100.npz", "holdout": "holdout_150.npz",
         "holdout_wide": "holdout_wide_50.npz"}
HR_BINS = np.arange(40, 181)
NO_PULSE = ("N0", "N1")


def load(split):
    if SMOKE:
        rng = np.random.default_rng(len(split))
        n = 256 if split == "train" else 128
        hr = rng.uniform(45, 160, n); sc = np.where(rng.random(n) < 0.15, "N0", "S0")
        hr[sc == "N0"] = np.nan
        return {"stmap": rng.standard_normal((n, 6, 16, 300)).astype(np.float16), "hr": hr,
                "scenario": sc}
    with np.load(CACHE / FILES[split], allow_pickle=False) as z:
        return {"stmap": z["stmap"].astype(np.float16), "hr": z["hr"].astype(np.float64),
                "scenario": z["scenario"].astype(str)}


def hr_target(hr, sigma=1.5):
    """141 Gaussian-smoothed HR bins + abstain class (one-hot abstain for no-pulse sessions)."""
    t = np.zeros(len(HR_BINS) + 1, np.float32)
    if hr is None or not np.isfinite(hr):
        t[-1] = 1.0
        return t
    g = np.exp(-0.5 * ((HR_BINS - hr) / sigma) ** 2)
    t[:-1] = g / g.sum()
    return t


tr, va, ho = load("train"), load("val"), load("holdout")
ytr = np.stack([hr_target(h) for h in tr["hr"]]); yva = np.stack([hr_target(h) for h in va["hr"]])
for k, d in (("train", tr), ("val", va), ("holdout", ho)):
    print(k, d["stmap"].shape, "no-pulse:", int(np.isin(d["scenario"], NO_PULSE).sum()))

## 2 · DataLoaders (L2, L3)

In [ ]:
def gpu_dls(xtr, ytr, xva, yva, bs=256):
    """L2: float16 tensors resident on the GPU, no worker processes, no batch transforms (L3)."""
    from fastai.data.core import DataLoaders
    from torch.utils.data import TensorDataset

    to = lambda a, dt: torch.as_tensor(a).to(DEVICE, dt)
    ds_tr = TensorDataset(to(xtr, torch.float16), to(ytr, ytr_dtype(ytr)))
    ds_va = TensorDataset(to(xva, torch.float16), to(yva, ytr_dtype(yva)))
    return DataLoaders.from_dsets(ds_tr, ds_va, bs=bs, num_workers=0, pin_memory=False,
                                  device=DEVICE)


def ytr_dtype(y):
    return torch.long if np.asarray(y).dtype.kind in "iu" else torch.float32


class Fp32Input(nn.Module):
    """Cast the float16 batch up once; to_fp16 then autocasts the convs back to half."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return self.m(x.float())


# L2: the reference recipe used bs=64, lr=3e-3; at bs=256 scale the LR by sqrt(256/64) = 2.
BS, LR = 256, 3e-3 * math.sqrt(256 / 64)

## 3 · Model (L4, L5)

In [ ]:
class PatchAttnSpec(nn.Module):
    """C3: per-patch dilated 1D CNN -> softmax patch attention -> 8 learned pulse signals ->
    fixed Hann DFT at exactly the 141 HR-bin frequencies (and 2f) -> 1D conv over frequency.
    ~0.13 M params, trained from scratch (L5: there is nothing pretrained to load)."""

    def __init__(self, in_ch=6, T=300, fs=10.0, width=64, k_sig=8, n_classes=142):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv1d(in_ch, width, 5, padding=2, bias=False), nn.BatchNorm1d(width), nn.ReLU(inplace=True),
            nn.Conv1d(width, width, 5, padding=4, dilation=2, bias=False), nn.BatchNorm1d(width), nn.ReLU(inplace=True),
            nn.Conv1d(width, width, 5, padding=8, dilation=4, bias=False), nn.BatchNorm1d(width), nn.ReLU(inplace=True),
        )
        self.score = nn.Sequential(nn.Linear(width, 32), nn.ReLU(inplace=True), nn.Linear(32, 1))
        self.mix = nn.Conv1d(width, k_sig, 1)
        t = np.arange(T) / fs
        win = np.hanning(T)
        freqs = np.concatenate([HR_BINS / 60.0, 2 * HR_BINS / 60.0])
        ang = 2 * np.pi * t[:, None] * freqs[None, :]
        self.register_buffer("C", torch.tensor(win[:, None] * np.cos(ang), dtype=torch.float32))
        self.register_buffer("S", torch.tensor(win[:, None] * np.sin(ang), dtype=torch.float32))
        self.k, self.nb = k_sig, len(HR_BINS)
        self.fhead = nn.Sequential(
            nn.Conv1d(2 * k_sig, width, 9, padding=4, bias=False), nn.BatchNorm1d(width), nn.ReLU(inplace=True),
            nn.Conv1d(width, width, 9, padding=8, dilation=2, bias=False), nn.BatchNorm1d(width), nn.ReLU(inplace=True),
        )
        self.bin_out = nn.Conv1d(width, 1, 1)
        self.abst = nn.Sequential(nn.Linear(2 * width, 32), nn.ReLU(inplace=True), nn.Linear(32, 1))

    def forward(self, x):  # x: (B, 6, P, T)
        B, C, P, T = x.shape
        h = self.enc(x.permute(0, 2, 1, 3).reshape(B * P, C, T))
        a = torch.softmax(self.score(h.abs().mean(2)).reshape(B, P), dim=1)
        s = (self.mix(h).reshape(B, P, self.k, T) * a[:, :, None, None]).sum(1)
        # L4: the DFT power spans many decades and the log needs eps=1e-6: keep it in FP32,
        # otherwise fp16 overflows/underflows and the loss goes NaN after a few epochs.
        with torch.autocast(device_type=x.device.type, enabled=False):
            s = s.float()
            s = s - s.mean(2, keepdim=True)
            pw = torch.matmul(s, self.C) ** 2 + torch.matmul(s, self.S) ** 2
            lp = torch.log(pw + 1e-6)
            feat = torch.cat([lp[..., : self.nb], lp[..., self.nb:]], 1)
            feat = feat - feat.mean(2, keepdim=True)
        z = self.fhead(feat)
        bins = self.bin_out(z)[:, 0]
        ab = self.abst(torch.cat([z.amax(2), z.mean(2)], 1))
        return torch.cat([bins, ab], 1)


def soft_ce(logits, target):
    return -(target * F.log_softmax(logits.float(), dim=1)).sum(1).mean()


print(sum(p.numel() for p in PatchAttnSpec().parameters()), "params")

## 4 · Train (L4, L7)

In [ ]:
from fastai.callback.schedule import fit_one_cycle  # noqa: F401
from fastai.learner import Learner

dls = gpu_dls(tr["stmap"], ytr, va["stmap"], yva, bs=BS)
torch.manual_seed(SEED)
learn = Learner(dls, Fp32Input(PatchAttnSpec()), loss_func=soft_ce)
if DEVICE.type == "cuda":
    learn = learn.to_fp16()
EPOCHS = 2 if SMOKE else 14   # L7: < 10 s/epoch on a T4 (~2 min/epoch on 4 CPU cores)
t0 = time.time()
learn.fit_one_cycle(EPOCHS, LR)
print(f"trained {EPOCHS} epochs in {time.time() - t0:.0f} s")
net = learn.model.m.float().eval()
torch.save(net.state_dict(), OUT / "C3_patch_attn_spec.pt")

## 4b · Decode, calibrate tau on val, score holdout once

In [ ]:
def logits_of(m, x, bs=512):
    m = m.to(DEVICE).eval(); out = []
    with torch.no_grad():
        for i in range(0, len(x), bs):
            out.append(m(torch.as_tensor(x[i:i + bs]).to(DEVICE).float()).cpu().numpy())
    return np.concatenate(out)


def decode_hr(logits, half=3):
    z = logits - logits.max(1, keepdims=True); p = np.exp(z); p /= p.sum(1, keepdims=True)
    pb = p[:, :141]; k = pb.argmax(1)
    hr, conf = np.empty(len(p)), np.empty(len(p))
    for i in range(len(p)):
        lo, hi = max(0, k[i] - half), min(141, k[i] + half + 1)
        w = pb[i, lo:hi]
        hr[i] = float((HR_BINS[lo:hi] * w).sum() / max(w.sum(), 1e-12)); conf[i] = float(w.sum())
    return hr, conf


def calib_tau(conf, is_nopulse, target=0.985):
    """Smallest cut at which `target` of val no-pulse sessions abstain (I30's tau rule)."""
    cn = np.sort(conf[is_nopulse])
    for cand in np.r_[0.0, cn]:
        if np.mean(cn < cand + 1e-12) >= target:
            return float(cand + 1e-12)
    return 1.0


def summary(d, hr, conf, tau):
    nop = np.isin(d["scenario"], NO_PULSE); ans = conf >= tau
    err = np.abs(hr - d["hr"])[~nop & ans]
    return {"tau": round(tau, 3), "coverage_pulse": round(float(ans[~nop].mean()), 3),
            "mae_answered": round(float(err.mean()), 2) if err.size else None,
            "within5": round(float((err <= 5).mean()), 3) if err.size else None,
            "nopulse_abstain": round(float((~ans[nop]).mean()), 3) if nop.any() else None}


hv, cv = decode_hr(logits_of(net, va["stmap"]))
hh, ch = decode_hr(logits_of(net, ho["stmap"]))
TAU = calib_tau(cv, np.isin(va["scenario"], NO_PULSE))
print("default tau (98.5 % val no-pulse abstain):", summary(ho, hh, ch, TAU))
# The shootout's val-tuned operating point was tau = 0.686 for its run; re-tune on val for yours.
json.dump({"tau_default": TAU}, open(OUT / "C3_tau.json", "w"))

In [ ]:
# Optional: C3 -> I30 cascade (C3 when conf >= tau, else I30 when it answers, else abstain).
# Attach service-intelligence/learning/eval_kit as a dataset (pbv_i30.py needs numpy + scipy).
try:
    sys.path.insert(0, "/kaggle/input/pranascan-eval-kit")
    import pbv_i30
    with np.load(CACHE / FILES["holdout"], allow_pickle=False) as z:
        t_s, roi = z["t_s"], z["roi"]
    i30 = [pbv_i30.estimate_hr(t, r) for t, r in zip(t_s, roi)]
    i30_hr = np.array([np.nan if r.hr_bpm is None else r.hr_bpm for r in i30])
    cas = np.where(ch >= TAU, hh, i30_hr)  # nan = abstain
    nop = np.isin(ho["scenario"], NO_PULSE); ans = np.isfinite(cas)
    err = np.abs(cas - ho["hr"])[~nop & ans]
    print({"cascade_coverage_pulse": round(float(ans[~nop].mean()), 3),
           "cascade_mae": round(float(err.mean()), 2) if err.size else None,
           "cascade_nopulse_abstain": round(float((~ans[nop]).mean()), 3) if nop.any() else None})
except (ImportError, FileNotFoundError, KeyError) as exc:
    print("cascade skipped:", type(exc).__name__, exc)

## 5 · Export: FP32 + conv-only static INT8 QDQ ONNX (L6, L7)

In [ ]:
def export_onnx(model, shape, path):
    """L6: TorchScript exporter (dynamo=False), opset 17, dynamic batch axis."""
    model = model.float().eval().cpu()
    torch.onnx.export(model, (torch.randn(*shape),), str(path), dynamo=False, opset_version=17,
                      input_names=["x"], output_names=["logits"],
                      dynamic_axes={"x": {0: "batch"}, "logits": {0: "batch"}})
    return path


def quantize_qdq(fp32_path, out_path, calib, op_types=None):
    """L7: static INT8 QDQ (per-channel weights). Dynamic INT8 is 5-30x slower on x86 CPUs."""
    from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                          quantize_static)
    from onnxruntime.quantization.shape_inference import quant_pre_process

    class Reader(CalibrationDataReader):
        def __init__(self, x):
            self.it = iter([{"x": x[i:i + 1].astype(np.float32)} for i in range(len(x))])

        def get_next(self):
            return next(self.it, None)

    pre = str(out_path).replace(".onnx", "_pre.onnx")
    quant_pre_process(str(fp32_path), pre, skip_symbolic_shape=True)
    kw = dict(quant_format=QuantFormat.QDQ, per_channel=True,
              activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
    if op_types:
        kw["op_types_to_quantize"] = op_types
    quantize_static(pre, str(out_path), Reader(calib), **kw)
    os.remove(pre)
    return out_path


def ort_logits(path, x, bs=128):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    return np.concatenate([s.run(None, {"x": x[i:i + bs].astype(np.float32)})[0]
                           for i in range(0, len(x), bs)])


def cpu1_latency_ms(path, shape, warm=30, runs=200):
    import onnxruntime as ort
    so = ort.SessionOptions(); so.intra_op_num_threads = 1; so.inter_op_num_threads = 1
    s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    x = np.random.default_rng(0).standard_normal(shape).astype(np.float32)
    for _ in range(warm):
        s.run(None, {"x": x})
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); s.run(None, {"x": x}); ts.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(ts))

In [ ]:
SHAPE = (1, 6, 16, 300)
fp32 = export_onnx(net, SHAPE, OUT / "C3_patch_attn_spec_fp32.onnx")
calib = va["stmap"][:: max(1, len(va["stmap"]) // 256)].astype(np.float32)
# Full INT8 breaks C3: 8-bit activations cannot hold the DFT power / log range. Quantize only
# Conv and Gemm and keep the spectral layer in FP32 (the shootout's int8_static_convonly).
int8 = quantize_qdq(fp32, OUT / "C3_patch_attn_spec_int8_static_convonly.onnx", calib,
                    op_types=["Conv", "Gemm"])
ref = logits_of(net, ho["stmap"][:64])
for p in (fp32, int8):
    lo = ort_logits(p, ho["stmap"].astype(np.float32))
    h, c = decode_hr(lo)
    print(p.name, f"{os.path.getsize(p) / 1e6:.3f} MB", f"cpu1 {cpu1_latency_ms(p, SHAPE):.2f} ms",
          f"max|dlogit| {np.abs(lo[:64] - ref).max():.3f}", summary(ho, h, c, TAU))

## Next

* Put `C3_patch_attn_spec_int8_static_convonly.onnx` in `learning/hf_space/models/` (or set
  `PRANASCAN_C3_ONNX`) and run the Space locally to compare it with POS and I30.
* Copy `tau_default` into the Space's C3 tau slider; keep the val-tuned value separate.
* Holdout is scored once. Re-tune tau on val only.